# Step 1: Simulate the marketplace and check the metrics

This notebook generates the simulated lending marketplace, stores it as warehouse-style tables in DuckDB, and computes the headline metrics two ways: in pandas and in SQL (PostgreSQL syntax). The two must match, which confirms the SQL metric layer is correct before any statistical analysis.

It also checks whether the simulation produces the intended pattern: CTR down, click-to-funded rate up, and funded loans per user up only modestly.

In [1]:
# Imports and paths. The notebook can be run from the project root or from notebooks/.
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.simulate import build_marketplace, simulate_outcomes, load_to_duckdb

SQL = ROOT / "sql"
DB_PATH = ROOT / "data" / "marketplace.duckdb"

In [2]:
# Build users and partners, then simulate offer exposure and outcomes.
# The partner table shows the payout-versus-approval tradeoff built into the marketplace;
# the tier mix shows that security_app traffic skews toward better credit.
users, partners, rng = build_marketplace()
impressions, truth = simulate_outcomes(users, partners, rng)

print(partners.sort_values("approval_mult").round(2).to_string(index=False))
print()
print(pd.crosstab(users["source"], users["tier_name"], normalize="index")
        [["excellent", "good", "fair", "poor"]].round(2))

partner_id  approval_mult  payout  appeal
       P07           0.70   375.0   -0.24
       P02           0.70   300.0    0.27
       P06           0.78   303.0   -0.33
       P08           0.90   300.0    0.10
       P00           1.19   153.0   -0.49
       P04           1.22   149.0    0.59
       P01           1.25   141.0    0.21
       P05           1.25   136.0    0.65
       P03           1.29   121.0    0.29
       P09           1.30   116.0    0.31

tier_name        excellent  good  fair  poor
source                                      
marketplace           0.15  0.35  0.30  0.20
partner_channel       0.20  0.35  0.28  0.17
security_app          0.30  0.40  0.20  0.10


In [3]:
# Headline metrics in pandas, plus the true lifts from ground truth.
# Expected pattern: CTR falls, click-to-funded rises a lot, funded per user rises modestly.
per_user = impressions.groupby("user_id")[["clicked", "applied", "funded"]].sum()
per_user["revenue"] = (impressions["payout"] * impressions["funded"]).groupby(impressions["user_id"]).sum()
per_user = per_user.join(users.set_index("user_id")["variant"])

g = per_user.groupby("variant")
pandas_summary = pd.DataFrame({
    "users": g.size(),
    "funded_per_user": g["funded"].mean(),
    "revenue_per_lead": g["revenue"].mean(),
    "ctr": g["clicked"].sum() / (g.size() * 5),
    "click_to_funded": g["funded"].sum() / g["clicked"].sum(),
    "approval_rate": g["funded"].sum() / g["applied"].sum(),
}).rename(index={0: "control", 1: "treatment"})
print(pandas_summary.round(4))

print()
for metric in ["funded", "revenue", "clicks"]:
    lift = truth[f"true_{metric}_t"].mean() / truth[f"true_{metric}_c"].mean() - 1
    print(f"True relative lift, {metric} per user: {lift:+.2%}")

            users  funded_per_user  revenue_per_lead     ctr  click_to_funded  \
variant                                                                         
control     99926           0.1566           28.7394  0.1514           0.2069   
treatment  100074           0.1603           29.2499  0.1400           0.2289   

           approval_rate  
variant                   
control           0.3766  
treatment         0.4178  

True relative lift, funded per user: +3.45%
True relative lift, revenue per user: +3.10%
True relative lift, clicks per user: -7.65%


In [4]:
# Store the simulation in DuckDB: users (observable columns only), partners,
# impressions (one row per offer shown), and ground truth in a separate table.
con = duckdb.connect(str(DB_PATH))
load_to_duckdb(con, users, partners, impressions, truth)
print(con.execute("SELECT COUNT(*) FROM users").fetchone()[0], "users")
print(con.execute("SELECT COUNT(*) FROM impressions").fetchone()[0], "impressions")

200000 users
1000000 impressions


In [5]:
# Compute the same metrics in SQL. The SQL files use PostgreSQL syntax and run on DuckDB.
# The comparison at the end must show matching values; a mismatch means a query bug.
con.execute((SQL / "01_user_metrics.sql").read_text())
sql_summary = con.execute((SQL / "02_arm_summary.sql").read_text()).df().set_index("arm")
print(sql_summary.round(4))

cols = ["users", "funded_per_user", "revenue_per_lead", "ctr", "click_to_funded", "approval_rate"]
match = np.allclose(sql_summary[cols].astype(float).to_numpy(),
                    pandas_summary[cols].astype(float).to_numpy())
print("\nSQL matches pandas:", match)

            users  funded_per_user  revenue_per_lead     ctr  click_to_funded  \
arm                                                                             
control     99926           0.1566           28.7394  0.1514           0.2069   
treatment  100074           0.1603           29.2499  0.1400           0.2289   

           approval_rate  
arm                       
control           0.3766  
treatment         0.4178  

SQL matches pandas: True


In [6]:
# Guardrail: partner lead share by arm. Large shifts toward a few partners would be
# a reason for caution even if funding rises.
lead_share = con.execute((SQL / "03_partner_lead_share.sql").read_text()).df()
lead_share["change"] = lead_share["treatment_share"] - lead_share["control_share"]
print(lead_share.round(3).to_string(index=False))

partner_id  control_share  treatment_share  change
       P01          0.047            0.043  -0.004
       P02          0.107            0.107   0.001
       P03          0.041            0.038  -0.004
       P04          0.299            0.307   0.008
       P05          0.308            0.316   0.008
       P06          0.000            0.001   0.000
       P07          0.013            0.017   0.003
       P08          0.146            0.139  -0.007
       P09          0.039            0.033  -0.006


In [7]:
# Sample ratio mismatch check: a chi-square test of users per arm against a 50/50 split.
# A very small p-value (e.g. below 0.001) would mean assignment or logging is broken.
srm = con.execute((SQL / "04_srm_counts.sql").read_text()).df()
chi2, p_value = stats.chisquare(srm["users"])
print(srm.to_string(index=False))
print(f"SRM chi-square p-value: {p_value:.3f}")

 variant  users
       0  99926
       1 100074
SRM chi-square p-value: 0.741


In [8]:
# Close the database connection. DuckDB allows only one process to hold the file open,
# so leaving it open blocks the next notebook from connecting.
con.close()